In [1]:
%cd ../..

c:\Users\ajaoo\Desktop\Projects\hospitalization_research


In [2]:
import pandas as pd
import numpy as np
import os
import time
import warnings
from tqdm.notebook import tqdm
from pathlib import Path
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRFRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error as mae, mean_squared_error as mse
import plotly.express as px
import plotly.graph_objects as go
from itertools import cycle
from src.utils import plotting_utils

np.random.seed(42)
tqdm.pandas()
import plotly.io as pio

pio.templates.default = "plotly_white"

c:\Users\ajaoo\Desktop\Projects\hospitalization_research\src\utils\data_utils.py:6: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
os.makedirs("data/output", exist_ok=True)
preprocessed = Path("data/NHS_region/Timeseries")
output = Path("data/output")

In [4]:
def format_plot(
    fig, legends=None, xlabel="Time", ylabel="Value", title="", font_size=15
):
    if legends:
        names = cycle(legends)
        fig.for_each_trace(lambda t: t.update(name=next(names)))
    fig.update_layout(
        autosize=False,
        width=900,
        height=500,
        title_text=title,
        title={"x": 0.5, "xanchor": "center", "yanchor": "top"},
        titlefont={"size": 20},
        legend_title=None,
        legend=dict(
            font=dict(size=font_size),
            orientation="h",
            yanchor="bottom",
            y=0.98,
            xanchor="right",
            x=1,
        ),
        yaxis=dict(
            title_text=ylabel,
            titlefont=dict(size=font_size),
            tickfont=dict(size=font_size),
        ),
        xaxis=dict(
            title_text=xlabel,
            titlefont=dict(size=font_size),
            tickfont=dict(size=font_size),
        ),
    )
    return fig

In [5]:
try:
    train_df = pd.read_csv(preprocessed / "featured_eng_train.csv")
    test_df = pd.read_csv(preprocessed / "featured_eng_test.csv")
    val_df = pd.read_csv(preprocessed / "featured_eng_val.csv")

except FileNotFoundError:
    print("File not found, please run the feature engineering notebook first")

## London region


In [6]:
train_df = train_df.drop(columns=["Unnamed: 0", "areaCode"])
test_df = test_df.drop(columns=["Unnamed: 0", "areaCode"])
val_df = val_df.drop(columns=["Unnamed: 0", "areaCode"])

In [8]:
train_df.head()

,areaName,areaType,date,covidOccupiedMVBeds,cumAdmissions,hospitalCases,newAdmissions,covidOccupiedMVBeds_trend,cumAdmissions_trend,hospitalCases_trend,...,covidOccupiedMVBeds_lag_1,covidOccupiedMVBeds_lag_7,covidOccupiedMVBeds_lag_14,covidOccupiedMVBeds_lag_21,covidOccupiedMVBeds_rolling_7_mean,covidOccupiedMVBeds_rolling_7_std,day_of_week,week_of_year,Month,Quarter
0,East of England,nhsRegion,2022-06-13,18,74381,431,62,16.857143,74643.571429,467.142857,...,25.0,15.0,23.0,25.0,16.857143,4.059087,0,24,6,2
1,East of England,nhsRegion,2022-06-12,19,74319,416,60,17.428571,74558.857143,452.571429,...,18.0,15.0,28.0,28.0,17.428571,4.035556,6,23,6,2
2,East of England,nhsRegion,2022-06-11,13,74259,413,54,17.142857,74480.571429,441.428571,...,19.0,15.0,23.0,27.0,17.142857,4.298394,5,23,6,2
3,East of England,nhsRegion,2022-06-10,9,74205,413,60,16.714286,74406.714286,432.000000,...,13.0,12.0,22.0,28.0,16.714286,4.990467,4,23,6,2
4,East of England,nhsRegion,2022-06-09,11,74145,399,77,16.000000,74334.714286,422.285714,...,9.0,16.0,17.0,28.0,16.000000,5.446712,3,23,6,2


In [9]:
train_df.columns

Index(['areaName', 'areaType', 'date', 'covidOccupiedMVBeds', 'cumAdmissions',
       'hospitalCases', 'newAdmissions', 'covidOccupiedMVBeds_trend',
       'cumAdmissions_trend', 'hospitalCases_trend', 'newAdmissions_trend',
       'covidOccupiedMVBeds_trend_diff',
       'covidOccupiedMVBeds_trend_seasonal_diff', 'covidOccupiedMVBeds_lag_1',
       'covidOccupiedMVBeds_lag_7', 'covidOccupiedMVBeds_lag_14',
       'covidOccupiedMVBeds_lag_21', 'covidOccupiedMVBeds_rolling_7_mean',
       'covidOccupiedMVBeds_rolling_7_std', 'day_of_week', 'week_of_year',
       'Month', 'Quarter'],
      dtype='object')

In [10]:
# Convert the date column to a datetime object
train_df["date"] = pd.to_datetime(train_df["date"])
test_df["date"] = pd.to_datetime(test_df["date"])
val_df["date"] = pd.to_datetime(val_df["date"])

In [11]:
# Filter data for London
sample_train_df = train_df.loc[
    train_df.areaName == "London",
    [
        'date',
        'cumAdmissions',
        'hospitalCases', 
        'newAdmissions', 
       'cumAdmissions_trend',
       'hospitalCases_trend',
       'newAdmissions_trend',
       'covidOccupiedMVBeds_trend_diff',
    'covidOccupiedMVBeds_lag_1',
       'covidOccupiedMVBeds_lag_7', 'covidOccupiedMVBeds_lag_14',
       'covidOccupiedMVBeds_lag_21', 'covidOccupiedMVBeds_rolling_7_mean',
       'covidOccupiedMVBeds_rolling_7_std', 'day_of_week', 'week_of_year',
       'Month', 'Quarter'
    ],
]
sample_test_df = test_df.loc[
    test_df.areaName == "London",
    [
        'date',
        'cumAdmissions',
        'hospitalCases', 
        'newAdmissions', 
       'cumAdmissions_trend',
       'hospitalCases_trend',
       'newAdmissions_trend',
       'covidOccupiedMVBeds_trend_diff',
    'covidOccupiedMVBeds_lag_1',
       'covidOccupiedMVBeds_lag_7', 'covidOccupiedMVBeds_lag_14',
       'covidOccupiedMVBeds_lag_21', 'covidOccupiedMVBeds_rolling_7_mean',
       'covidOccupiedMVBeds_rolling_7_std', 'day_of_week', 'week_of_year',
       'Month', 'Quarter'
    ],
]

In [12]:
# Convert date to datetime and set as index
sample_train_df["date"] = pd.to_datetime(sample_train_df["date"])
sample_test_df["date"] = pd.to_datetime(sample_test_df["date"])

sample_train_df.set_index("date", inplace=True)
sample_test_df.set_index("date", inplace=True)

In [13]:
# Separate features and target
train_features = sample_train_df.drop(columns=["covidOccupiedMVBeds_trend_diff"])
train_target = sample_train_df["covidOccupiedMVBeds_trend_diff"]

test_features = sample_test_df.drop(columns=["covidOccupiedMVBeds_trend_diff"])
test_target = sample_test_df["covidOccupiedMVBeds_trend_diff"]

In [16]:
class ModelConfig:
    def __init__(self, model, name, normalize=False, fill_missing=False):
        self.model = model
        self.name = name
        self.normalize = normalize
        self.fill_missing = fill_missing


class LogTime:
    from time import time

    def __enter__(self):
        self.start_time = self.time()
        print("Starting operation...")

    def __exit__(self, type, value, traceback):
        elapsed_time = self.time() - self.start_time
        print(f"Operation completed in {elapsed_time} seconds.")

In [17]:
def preprocess_data(train_features, test_features, config):
    if config.fill_missing:
        train_features = train_features.fillna(method="ffill")
        test_features = test_features.fillna(method="ffill")

    if config.normalize:
        scaler = StandardScaler()
        train_features = scaler.fit_transform(train_features)
        test_features = scaler.transform(test_features)

    return train_features, test_features

In [18]:
def mase(actual, predicted, insample_actual):
    mae_insample = np.mean(np.abs(np.diff(insample_actual)))
    mae_outsample = np.mean(np.abs(actual - predicted))
    return mae_outsample / mae_insample


def forecast_bias(actual, predicted):
    return np.mean(predicted - actual)

In [19]:
def evaluate_model(config, train_features, train_target, test_features, test_target):
    # Preprocess the data
    train_features, test_features = preprocess_data(
        train_features, test_features, config
    )

    # Fit the model
    config.model.fit(train_features, train_target)

    # Predict
    y_pred = config.model.predict(test_features)

    # Compute metrics
    metrics = {
        "Algorithm": config.name,
        "MAE": mae(test_target, y_pred),
        "MSE": mse(test_target, y_pred),
        "MASE": mase(test_target, y_pred, train_target),
        "Forecast Bias": forecast_bias(test_target, y_pred),
    }

    # Return predictions and metrics
    return y_pred, metrics

In [20]:
def plot_forecast(pred_df, forecast_columns, forecast_display_names=None):
    if forecast_display_names is None:
        forecast_display_names = forecast_columns
    else:
        assert len(forecast_columns) == len(forecast_display_names)
    mask = ~pred_df[forecast_columns[0]].isnull()
    colors = [
        "rgba(" + ",".join([str(c) for c in plotting_utils.hex_to_rgb(c)]) + ",<alpha>)"
        for c in px.colors.qualitative.Plotly
    ]
    act_color = colors[0]
    colors = cycle(colors[1:])
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=pred_df[mask].index,
            y=pred_df[mask].covidOccupiedMVBeds_trend_diff,
            mode="lines",
            line=dict(color=act_color.replace("<alpha>", "0.9")),
            name="7 day-moving average MVBeds_trends",
        )
    )
    for col, display_col in zip(forecast_columns, forecast_display_names):
        fig.add_trace(
            go.Scatter(
                x=pred_df[mask].index,
                y=pred_df.loc[mask, col],
                mode="lines",
                line=dict(dash="dot", color=next(colors).replace("<alpha>", "1")),
                name=display_col,
            )
        )
    return fig

def highlight_abs_min(s, props=""):
    return np.where(s == np.nanmin(np.abs(s.values)), props, "")

In [21]:
# Record metrics
metric_record = []

## Running ML models on a single region (London) household


### Linear Regression


In [22]:
# Define the model configuration
lr_model_config = ModelConfig(
    model=LinearRegression(),
    name="Linear Regression",
    normalize=True,
    fill_missing=True,
)

with LogTime() as timer:
    start_time = time.time()
    y_pred, metrics = evaluate_model(
        lr_model_config,
        train_features,
        train_target,
        test_features,
        test_target,
    )

elapsed_time = time.time() - start_time
metrics["Time Elapsed"] = elapsed_time

metric_record.append(metrics)

# Create prediction DataFrame
pred_df = pd.DataFrame(
    {
        "covidOccupiedMVBeds_trend_diff": pd.concat([train_target, test_target]),
        lr_model_config.name: pd.Series(y_pred, index=test_target.index),
    }
)

Starting operation...
Operation completed in 0.044103145599365234 seconds.


C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  train_features = train_features.fillna(method="ffill")
C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  test_features = test_features.fillna(method="ffill")


In [23]:
# Plot forecast
fig = plot_forecast(
    pred_df,
    forecast_columns=[lr_model_config.name],
    forecast_display_names=[lr_model_config.name],
)
fig = format_plot(
    fig,
    title=f"{lr_model_config.name}: MAE: {metrics['MAE']:.4f} | MSE: {metrics['MSE']:.4f} | MASE: {metrics['MASE']:.4f} | Bias: {metrics['Forecast Bias']:.4f}",
)
fig.update_xaxes(type="date", range=["2022-012-01", "2023-04-01"])
# fig.write_image("imgs/chapter_7/lin_reg.png")
fig.show()

In [24]:
# If you want to plot feature importance and the model used provides it
if hasattr(lr_model_config.model, "coef_"):
    feat_df = pd.DataFrame(
        {"feature": train_features.columns, "importance": lr_model_config.model.coef_}
    )
    fig = px.bar(feat_df.head(15), x="feature", y="importance")
    format_plot(
        fig,
        xlabel="Features",
        ylabel="Importance",
        title=f"Feature Importance - {lr_model_config.name}",
        font_size=12,
    )
    # fig.write_image("imgs/chapter_7/lin_reg_fimp.png")
    fig.show()

### Random Forest


In [25]:
# Define the Random Forest model configuration
rf_model_config = ModelConfig(
    model=RandomForestRegressor(n_estimators=100, random_state=42, max_depth=4),
    name="Random Forest",
    # RandomForest is not affected by normalization
    normalize=False,
    fill_missing=True,
)


with LogTime() as timer:
    start_time = time.time()
    y_pred, metrics = evaluate_model(
        rf_model_config,
        train_features,
        train_target,
        test_features,
        test_target,
    )

# Compute elapsed time
elapsed_time_rf = time.time() - start_time
metrics["Time Elapsed"] = elapsed_time_rf

# Append the metrics to the record
metric_record.append(metrics)

# Create prediction DataFrame for Random Forest
pred_df[rf_model_config.name] = pd.Series(y_pred, index=test_target.index)

Starting operation...


C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:3: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:4: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



Operation completed in 0.34228515625 seconds.


In [28]:
# Plot forecast
fig = plot_forecast(
    pred_df,
    forecast_columns=[rf_model_config.name],
    forecast_display_names=[rf_model_config.name],
)
fig = format_plot(
    fig,
    title=f"{rf_model_config.name}: MAE: {metrics['MAE']:.4f} | MSE: {metrics['MSE']:.4f} | MASE: {metrics['MASE']:.4f} | Bias: {metrics['Forecast Bias']:.4f}",
)
fig.update_xaxes(type="date", range=["2022-012-01", "2023-04-01"])
# fig.write_image("imgs/chapter_7/rf.png")
fig.show()

In [29]:
# Plot feature importance for Random Forest
if hasattr(rf_model_config.model, "feature_importances_"):
    feat_df_rf = pd.DataFrame(
        {
            "feature": train_features.columns,
            "importance": rf_model_config.model.feature_importances_,
        }
    )
    fig = px.bar(feat_df_rf.head(20), x="feature", y="importance")
    format_plot(
        fig,
        xlabel="Features",
        ylabel="Importance",
        title=f"Feature Importance - {rf_model_config.name}",
        font_size=12,
    )
    fig.show()

### Ridge Regression (L2)


In [30]:
rr_model_config = ModelConfig(
    model=RidgeCV(),
    name="Ridge Regression",
    # RidgeCV is sensitive to normalized data
    normalize=True,
    # RidgeCV does not handle missing values
    fill_missing=True,
)

with LogTime() as timer:
    start_time = time.time()
    y_pred, metrics = evaluate_model(
        rr_model_config,
        train_features,
        train_target,
        test_features,
        test_target,
    )

# Compute elapsed time
elapsed_time = time.time() - start_time
metrics["Time Elapsed"] = elapsed_time

# Append the metrics to the record
metric_record.append(metrics)

# Create prediction DataFrame for Random Forest
pred_df[rr_model_config.name] = pd.Series(y_pred, index=test_target.index)

Starting operation...
Operation completed in 0.041423797607421875 seconds.


C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:3: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:4: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



In [32]:
# Plot forecast
fig = plot_forecast(
    pred_df,
    forecast_columns=[rr_model_config.name],
    forecast_display_names=[rr_model_config.name],
)

fig = format_plot(
    fig,
    title=f"{rr_model_config.name}: MAE: {metrics['MAE']:.4f} | MSE: {metrics['MSE']:.4f} | MASE: {metrics['MASE']:.4f} | Bias: {metrics['Forecast Bias']:.4f}",
)
fig.update_xaxes(type="date", range=["2022-012-01", "2023-04-01"])
# fig.write_image("imgs/chapter_7/ridge.png")
fig.show()

In [33]:
# Plot feature importance for RidgeCV
if hasattr(rr_model_config.model, "coef_"):
    feat_df_rr_model = pd.DataFrame(
        {"feature": train_features.columns, "importance": rr_model_config.model.coef_}
    )
    fig = px.bar(feat_df_rr_model.head(20), x="feature", y="importance")
    format_plot(
        fig,
        xlabel="Features",
        ylabel="Importance",
        title=f"Feature Importance - {rr_model_config.name}",
        font_size=12,
    )
    fig.show()

### XGB Random Forest model


In [34]:
XG_model_config = ModelConfig(
    model=XGBRFRegressor(random_state=42, max_depth=4),
    name="XGB Random Forest",
    # XGBRF is not affected by normalization
    normalize=False,
    # XGBRF handles missing values
    fill_missing=False,
)

with LogTime() as timer:
    start_time = time.time()
    y_pred, metrics = evaluate_model(
        XG_model_config,
        train_features,
        train_target,
        test_features,
        test_target,
    )

# Compute elapsed time
elapsed_time = time.time() - start_time
metrics["Time Elapsed"] = elapsed_time

# Append the metrics to the record
metric_record.append(metrics)

# Create prediction DataFrame for Random Forest
pred_df[XG_model_config.name] = pd.Series(y_pred, index=test_target.index)

Starting operation...
Operation completed in 0.22397136688232422 seconds.


In [36]:
# Plot forecast
fig = plot_forecast(
    pred_df,
    forecast_columns=[XG_model_config.name],
    forecast_display_names=[XG_model_config.name],
)

fig = format_plot(
    fig,
    title=f"{XG_model_config.name}: MAE: {metrics['MAE']:.4f} | MSE: {metrics['MSE']:.4f} | MASE: {metrics['MASE']:.4f} | Bias: {metrics['Forecast Bias']:.4f}",
)
fig.update_xaxes(type="date", range=["2022-012-01", "2023-04-01"])
# fig.write_image("imgs/chapter_7/xgb.png")
fig.show()


In [37]:
# Plot feature importance for XGBRFRegressor
if hasattr(XG_model_config.model, "feature_importances_"):
    feat_df_xgbrf = pd.DataFrame(
        {
            "feature": train_features.columns,
            "importance": XG_model_config.model.feature_importances_,
        }
    )
    fig = px.bar(feat_df_xgbrf.head(20), x="feature", y="importance")
    format_plot(
        fig,
        xlabel="Features",
        ylabel="Importance",
        title=f"Feature Importance - {XG_model_config.name}",
        font_size=12,
    )
    fig.show()

In [38]:
from lightgbm import LGBMRegressor

In [39]:
LG_model_config = ModelConfig(
    model=LGBMRegressor(random_state=42),
    name="LightGBM",
    # LightGBM is not affected by normalization
    normalize=False,
    # LightGBM handles missing values
    fill_missing=False,
)
with LogTime() as timer:
    start_time = time.time()
    y_pred, metrics = evaluate_model(
        LG_model_config,
        train_features,
        train_target,
        test_features,
        test_target,
    )

# Compute elapsed time
elapsed_time = time.time() - start_time
metrics["Time Elapsed"] = elapsed_time

# Append the metrics to the record
metric_record.append(metrics)

# Create prediction DataFrame for Random Forest
pred_df[LG_model_config.name] = pd.Series(y_pred, index=test_target.index)

Starting operation...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000669 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3127
[LightGBM] [Info] Number of data points in the train set: 803, number of used features: 16
[LightGBM] [Info] Start training from score 0.952144
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

In [41]:
# Plot forecast
fig = plot_forecast(
    pred_df,
    forecast_columns=[LG_model_config.name],
    forecast_display_names=[LG_model_config.name],
)
fig = format_plot(
    fig,
    title=f"{LG_model_config.name}: MAE: {metrics['MAE']:.4f} | MSE: {metrics['MSE']:.4f} | MASE: {metrics['MASE']:.4f} | Bias: {metrics['Forecast Bias']:.4f}",
)
fig.update_xaxes(type="date", range=["2022-012-01", "2023-04-01"])
# fig.write_image("imgs/chapter_7/lin_reg.png")
fig.show()

In [43]:
# Plot forecast
forecast_columns_rf = [
    lr_model_config.name,
    rr_model_config.name,
    LG_model_config.name
]
fig = plot_forecast(
    pred_df,
    forecast_columns=forecast_columns_rf,
    forecast_display_names=forecast_columns_rf,
)
fig.show()

In [44]:
formatted = pd.DataFrame(metric_record).style.format(
    {"MAE": "{:.4f}", "MSE": "{:.4f}", "MASE": "{:.4f}", "Forecast Bias": "{:.2f}%"}
)
formatted = formatted.highlight_min(
    color="lightgreen", subset=["MAE", "MSE", "MASE"]
).apply(
    highlight_abs_min,
    props="color:black;background-color:lightgreen",
    axis=0,
    subset=["Forecast Bias"],
)
formatted

,Algorithm,MAE,MSE,MASE,Forecast Bias,Time Elapsed
0,Linear Regression,0.7061,0.8343,0.4925,0.21%,0.044103
1,Random Forest,1.0664,1.7428,0.7437,0.29%,0.342285
2,Ridge Regression,0.7469,0.9275,0.5209,0.26%,0.041424
3,XGB Random Forest,1.0540,1.7109,0.7351,0.23%,0.224970
4,LightGBM,0.8327,1.2061,0.5807,0.30%,0.397080


### Evaluation of ML Forecast


### Running ML forecast for all regions


In [45]:
try:
    train_df = pd.read_csv(preprocessed / "featured_eng_train.csv")
    test_df = pd.read_csv(preprocessed / "featured_eng_test.csv")
    val_df = pd.read_csv(preprocessed / "featured_eng_val.csv")

except FileNotFoundError:
    print("File not found, please run the feature engineering notebook first")

In [46]:
train_df = train_df.drop(columns=["Unnamed: 0", "areaCode"])
test_df = test_df.drop(columns=["Unnamed: 0", "areaCode"])
val_df = val_df.drop(columns=["Unnamed: 0", "areaCode"])

In [47]:
# Get unique regions
unique_regions = train_df["areaName"].unique()

# Initialize the metric record for all regions
metric_record_all_regions = []

In [48]:
models_config = [
    ModelConfig(
        model=LinearRegression(), 
        name="Linear regression",
        normalize=True, 
        fill_missing=True
    ),
    ModelConfig(
        model=XGBRFRegressor(random_state=42, max_depth=5, subsample= 0.8, learning_rate = 1),
        name="XGB Random Forest",
        normalize=False,
        fill_missing=False,
    ),
    ModelConfig(
        model=RidgeCV(alphas=(0.1, 1.0, 10.0), fit_intercept=True, scoring=None, cv=5),
        name="Ridge Regression",
        normalize=True,
        fill_missing=True,
    ),
]

In [49]:
# Loop through each unique region in the areaName column

for region in unique_regions:
    # Filter data for the specific region
    train_region_df = train_df[train_df["areaName"] == region]
    test_region_df = test_df[test_df["areaName"] == region]
    val_region_df = val_df[val_df["areaName"] == region]

    # Perform the same operations for training, testing, and validation data
    for data_type, region_df in zip(
        ["val"], [val_region_df]
    ):
        # Convert date to datetime and set as index
        region_df["date"] = pd.to_datetime(region_df["date"])
        region_df.set_index("date", inplace=True)

        # Separate features and target
        features = region_df.drop(
            columns=["areaName", "areaType", "covidOccupiedMVBeds_trend_diff", "covidOccupiedMVBeds_trend_seasonal_diff", "covidOccupiedMVBeds", "covidOccupiedMVBeds_trend"]
        )
        target = region_df["covidOccupiedMVBeds_trend_diff"]

        # Initialize metric records for the region
        metric_record_region = []

        # Loop through each model configuration and evaluate
        for model_config in models_config:
            y_pred, metrics = evaluate_model(
                model_config,
                features,
                target,
                test_features,  # Modify as per your validation/testing strategy
                test_target,  # Modify as per your validation/testing strategy
            )

            # Append the region name and data type to the metrics
            metrics["Region"] = region
            metrics["Algorithm"] = model_config.name
            metrics["Data Type"] = data_type

            # Append the metrics to the record
            metric_record_region.append(metrics)

        # Print metrics for the region
        formatted_new = pd.DataFrame(metric_record_region).style.format(
            {
                "MAE": "{:.4f}",
                "MSE": "{:.4f}",
                "MASE": "{:.4f}",
                "Forecast Bias": "{:.2f}%",
            }
        )
        formatted_new = formatted_new.highlight_min(
            color="lightgreen", subset=["MAE", "MSE", "MASE"]
        ).apply(
            highlight_abs_min,
            props="color:black;background-color:lightgreen",
            axis=0,
            subset=["Forecast Bias"],
        )
        print(f"Metrics for region: {region} - {data_type}")
        display(formatted)

        # Combine metrics for all regions
        metric_record_all_regions.extend(metric_record_region)

Metrics for region: East of England - val


C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:3: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:4: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:3: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:4: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



,Algorithm,MAE,MSE,MASE,Forecast Bias,Time Elapsed
0,Linear Regression,0.7061,0.8343,0.4925,0.21%,0.044103
1,Random Forest,1.0664,1.7428,0.7437,0.29%,0.342285
2,Ridge Regression,0.7469,0.9275,0.5209,0.26%,0.041424
3,XGB Random Forest,1.0540,1.7109,0.7351,0.23%,0.224970
4,LightGBM,0.8327,1.2061,0.5807,0.30%,0.397080


Metrics for region: North East and Yorkshire - val


C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:3: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:4: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:3: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:4: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



...

Metrics for region: London - val


,Algorithm,MAE,MSE,MASE,Forecast Bias,Time Elapsed
0,Linear Regression,0.7061,0.8343,0.4925,0.21%,0.044103
1,Random Forest,1.0664,1.7428,0.7437,0.29%,0.342285
2,Ridge Regression,0.7469,0.9275,0.5209,0.26%,0.041424
3,XGB Random Forest,1.0540,1.7109,0.7351,0.23%,0.224970
4,LightGBM,0.8327,1.2061,0.5807,0.30%,0.397080


C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:3: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:4: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:3: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\ajaoo\AppData\Local\Temp\ipykernel_28540\3447116977.py:4: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



Metrics for region: South West - val


,Algorithm,MAE,MSE,MASE,Forecast Bias,Time Elapsed
0,Linear Regression,0.7061,0.8343,0.4925,0.21%,0.044103
1,Random Forest,1.0664,1.7428,0.7437,0.29%,0.342285
2,Ridge Regression,0.7469,0.9275,0.5209,0.26%,0.041424
3,XGB Random Forest,1.0540,1.7109,0.7351,0.23%,0.224970
4,LightGBM,0.8327,1.2061,0.5807,0.30%,0.397080


In [50]:
# Print summary metrics for all regions
formatted_summary = pd.DataFrame(metric_record_all_regions).style.format(
    {"MAE": "{:.4f}", "MSE": "{:.4f}", "MASE": "{:.4f}", "Forecast Bias": "{:.2f}%"}
)
formatted_summary = formatted_summary.highlight_min(
    color="lightgreen", subset=["MAE", "MSE", "MASE"]
).apply(
    highlight_abs_min,
    props="color:black;background-color:lightgreen",
    axis=0,
    subset=["Forecast Bias"],
)
print("Summary metrics for all regions:")
display(formatted_summary)

Summary metrics for all regions:


,Algorithm,MAE,MSE,MASE,Forecast Bias,Region,Data Type
0,Linear regression,9.9886,101.7464,22.6212,-9.99%,East of England,val
1,XGB Random Forest,1.1516,2.1748,2.6081,-0.53%,East of England,val
2,Ridge Regression,6.1374,38.3745,13.8995,-6.14%,East of England,val
3,Linear regression,0.7311,0.8754,1.4462,-0.07%,North East and Yorkshire,val
4,XGB Random Forest,1.2165,2.4206,2.4065,-0.76%,North East and Yorkshire,val
5,Ridge Regression,0.9118,1.1942,1.8038,-0.83%,North East and Yorkshire,val
6,Linear regression,8.5743,105.0730,16.8622,8.56%,South East,val
7,XGB Random Forest,1.3102,2.8314,2.5765,-0.96%,South East,val
8,Ridge Regression,5.8962,50.6114,11.5955,5.77%,South East,val
9,Linear regression,3.2263,11.9352,7.5456,-3.21%,North West,val


In [51]:
fig = px.histogram(
    metric_record_all_regions,
    x="MASE",
    color="Algorithm",
    pattern_shape="Algorithm",
    marginal="box",
    nbins=50,
    barmode="overlay",
    histnorm="probability density",
)
fig = format_plot(
    fig,
    xlabel="MASE",
    ylabel="Probability Density",
    title="Distribution of MASE in the dataset",
)
fig.update_layout(xaxis_range=[0, 10.5])
# fig.write_image("imgs/chapter_7/mase_dist.png")
fig.show()

In [52]:
metric_record_all_regions

[{'Algorithm': 'Linear regression',
  'MAE': 9.98858563162394,
  'MSE': 101.74640865533522,
  'MASE': 22.621208636324795,
  'Forecast Bias': -9.98858563162394,
  'Region': 'East of England',
  'Data Type': 'val'},
 {'Algorithm': 'XGB Random Forest',
  'MAE': 1.1516311524093754,
  'MSE': 2.1747859561875873,
  'MASE': 2.608105845162408,
  'Forecast Bias': -0.5340708631667352,
  'Region': 'East of England',
  'Data Type': 'val'},
 {'Algorithm': 'Ridge Regression',
  'MAE': 6.137445433948983,
  'MSE': 38.3745434631467,
  'MASE': 13.899508776884456,
  'Forecast Bias': -6.137445433948983,
  'Region': 'East of England',
  'Data Type': 'val'},
 {'Algorithm': 'Linear regression',
  'MAE': 0.7310639391484814,
  'MSE': 0.8753915601934203,
  'MASE': 1.4462351839676473,
  'Forecast Bias': -0.0714623061082912,
  'Region': 'North East and Yorkshire',
  'Data Type': 'val'},
 {'Algorithm': 'XGB Random Forest',
  'MAE': 1.2164934316561333,
  'MSE': 2.420617294135361,
  'MASE': 2.406541353928436,
  'Fore